# ASD detection: Face + EEG fusion, end-to-end (Colab)
**Before you start:** Runtime > Change runtime type > **T4 GPU**.
Upload the repository folder structure to Google Drive, e.g. `MyDrive/asd_fusion/`:
- `scripts/` (Python scripts), `results/` (clean face manifest and EEG feature file), and `data/raw/faces/` (face dataset archive)
- `eeg_windows_v2.npz` in `results/` (output of prepare_eeg_v2.py)

Run the cells top to bottom. Each step prints a report and saves a file.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/asd_fusion'   # <-- change if your folder is named differently
!mkdir -p /content/work/scripts /content/work/results && cp "$PROJECT"/scripts/*.py /content/work/scripts/ && cp "$PROJECT"/results/*.csv "$PROJECT"/results/*.npz /content/work/results/ 2>/dev/null; ls /content/work/results
%cd /content/work/results
!pip install -q PyWavelets
import torch; print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. Unzip the face dataset to the fast local disk (reading from Drive during training is very slow)

In [ ]:
!mkdir -p /content/faces_raw && unzip -q -o "$PROJECT/data/raw/faces/Autistic_Children_Facial_Image.zip" -d /content/faces_raw   # <-- adjust zip name
!find /content/faces_raw -maxdepth 3 -type d | head -20

In [ ]:
# Set FACES_ROOT to the folder that directly contains train/ valid/ test/ (read it off the listing above)
FACES_ROOT = '/content/faces_raw/Autistic_Children_Facial_Image'
import os; print(os.listdir(FACES_ROOT))

## 2. Shortcut test: can photo properties alone predict the label?

In [ ]:
!python ../scripts/face_shortcut_test.py --root "$FACES_ROOT" --manifest faces_clean_manifest.csv
# Expected counts after the manifest: train 1258/1244, valid 99/98, test 98/96. If they differ, stop and check the manifest columns.

## 3. Face CNN (main run, RGB)  ~ 5-10 min on a T4

In [ ]:
!python ../scripts/face_cnn.py --root "$FACES_ROOT" --manifest faces_clean_manifest.csv --variant rgb

## 4. Robustness run (grayscale + histogram equalised). A big drop vs RGB means colour/brightness cues were helping.

In [ ]:
!python ../scripts/face_cnn.py --root "$FACES_ROOT" --manifest faces_clean_manifest.csv --variant grayeq

## 5. Grad-CAM: is the model looking at the face?

In [ ]:
!python ../scripts/gradcam.py --root "$FACES_ROOT" --manifest faces_clean_manifest.csv --variant rgb
from IPython.display import Image, display; display(Image('gradcam_rgb.png'))

## 6. Fusion + ablation table (EEG variant A = primary)

In [ ]:
!python ../scripts/fusion_eval.py --eeg eeg_windows_v2.npz --variant A --face_out face_out_rgb.npz --face_emb face_emb.npz --reps 20

In [ ]:
# Optional: same fusion with the grayscale/equalised face model, and with another EEG variant you declared up front
!python ../scripts/fusion_eval.py --eeg eeg_windows_v2.npz --variant A --face_out face_out_grayeq.npz --face_emb face_emb.npz --reps 20 --tag A_grayeq

## 7. Save everything back to Drive

In [ ]:
!mkdir -p "$PROJECT/results" && cp *.txt *.npz *.png *.pt "$PROJECT/results/" 2>/dev/null; ls "$PROJECT/results" 